# 05 SQL Analysis

This notebook creates a SQLite database from the cleaned NHANES dataset
and demonstrates SQL querying for health data analysis.

Input: cleaned_data.csv (from notebook 02)
Database: nhanes.db (SQLite)
SQL files: sql/create_tables.sql, sql/queries.sql

In [12]:
import pandas as pd
import sqlite3

# Load cleaned data
df = pd.read_csv('../data/cleaned_data.csv')

# Create SQLite database
conn = sqlite3.connect('../data/nhanes.db')

# Write dataframe to SQL table
df.to_sql('nhanes_clean', conn, if_exists='replace', index=False)

print("Database created successfully.")
print("Rows inserted:", pd.read_sql("SELECT COUNT(*) as n FROM nhanes_clean", conn).iloc[0,0])

Database created successfully.
Rows inserted: 1347


In [13]:
# Query 1: Basic descriptive statistics by functional limitation status
query1 = """
SELECT 
    functional_limitation,
    COUNT(*) as n,
    ROUND(AVG(BMXBMI), 2) as mean_bmi,
    ROUND(AVG(sitting_hours), 2) as mean_sitting_hours,
    ROUND(AVG(RIDAGEYR), 2) as mean_age
FROM nhanes_clean
GROUP BY functional_limitation
"""

result1 = pd.read_sql(query1, conn)
result1['functional_limitation'] = result1['functional_limitation'].map({0: 'No Limitation', 1: 'Has Limitation'})
print(result1)

  functional_limitation    n  mean_bmi  mean_sitting_hours  mean_age
0         No Limitation  470     28.91                5.09     53.10
1        Has Limitation  877     32.18                5.86     52.97


In [14]:
# Query 2: Functional limitation rate by age group
query2 = """
SELECT 
    CASE 
        WHEN RIDAGEYR BETWEEN 20 AND 35 THEN '20-35'
        WHEN RIDAGEYR BETWEEN 36 AND 50 THEN '36-50'
        WHEN RIDAGEYR BETWEEN 51 AND 65 THEN '51-65'
    END as age_group,
    COUNT(*) as n,
    ROUND(AVG(functional_limitation) * 100, 1) as limitation_rate_pct
FROM nhanes_clean
GROUP BY age_group
ORDER BY age_group
"""

result2 = pd.read_sql(query2, conn)
print(result2)

  age_group    n  limitation_rate_pct
0     20-35  182                 49.5
1     36-50  228                 81.6
2     51-65  937                 64.1


In [15]:
# Query 3: JOIN demonstration - sedentary behavior and BMI risk profile
query3 = """
SELECT 
    CASE 
        WHEN BMXBMI < 25 THEN 'Normal'
        WHEN BMXBMI BETWEEN 25 AND 30 THEN 'Overweight'
        ELSE 'Obese'
    END as bmi_category,
    CASE
        WHEN sitting_hours < 4 THEN 'Low'
        WHEN sitting_hours BETWEEN 4 AND 8 THEN 'Moderate'
        ELSE 'High'
    END as sedentary_level,
    COUNT(*) as n,
    ROUND(AVG(functional_limitation) * 100, 1) as limitation_rate_pct
FROM nhanes_clean
GROUP BY bmi_category, sedentary_level
ORDER BY bmi_category, sedentary_level
"""

result3 = pd.read_sql(query3, conn)
print(result3)

  bmi_category sedentary_level    n  limitation_rate_pct
0       Normal            High   41                 70.7
1       Normal             Low  103                 48.5
2       Normal        Moderate  146                 56.8
3        Obese            High  137                 80.3
4        Obese             Low  164                 66.5
5        Obese        Moderate  339                 74.3
6   Overweight            High   58                 51.7
7   Overweight             Low  154                 56.5
8   Overweight        Moderate  205                 62.0


In [16]:
# Query 4: Functional limitation rate by arthritis status
query4 = """
SELECT 
    arthritis,
    COUNT(*) as n,
    ROUND(AVG(functional_limitation) * 100, 1) as limitation_rate_pct,
    ROUND(AVG(BMXBMI), 2) as mean_bmi,
    ROUND(AVG(sitting_hours), 2) as mean_sitting_hours
FROM nhanes_clean
GROUP BY arthritis
"""

result4 = pd.read_sql(query4, conn)
result4['arthritis'] = result4['arthritis'].map({0: 'No Arthritis', 1: 'Has Arthritis'})
print(result4)

       arthritis    n  limitation_rate_pct  mean_bmi  mean_sitting_hours
0   No Arthritis  772                 51.0     30.02                5.46
1  Has Arthritis  575                 84.0     32.41                5.77


In [17]:
# Save query results to visualizations for reference
print("=== Query 1: Descriptive Statistics ===")
print(result1.to_string(index=False))

print("\n=== Query 2: Age Group Analysis ===")
print(result2.to_string(index=False))

print("\n=== Query 3: BMI x Sedentary Risk Profile ===")
print(result3.to_string(index=False))

print("\n=== Query 4: Arthritis and Functional Limitation ===")
print(result4.to_string(index=False))

# Close database connection
conn.close()
print("\nDatabase connection closed.")

=== Query 1: Descriptive Statistics ===
functional_limitation   n  mean_bmi  mean_sitting_hours  mean_age
        No Limitation 470     28.91                5.09     53.10
       Has Limitation 877     32.18                5.86     52.97

=== Query 2: Age Group Analysis ===
age_group   n  limitation_rate_pct
    20-35 182                 49.5
    36-50 228                 81.6
    51-65 937                 64.1

=== Query 3: BMI x Sedentary Risk Profile ===
bmi_category sedentary_level   n  limitation_rate_pct
      Normal            High  41                 70.7
      Normal             Low 103                 48.5
      Normal        Moderate 146                 56.8
       Obese            High 137                 80.3
       Obese             Low 164                 66.5
       Obese        Moderate 339                 74.3
  Overweight            High  58                 51.7
  Overweight             Low 154                 56.5
  Overweight        Moderate 205                 62.